<a href="https://colab.research.google.com/github/nikitask14/pytorch-engineering-to-federated-learning/blob/main/Sitting20_Copying_models_safely_and_checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import copy

In [2]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.layer2(x)

    return x


#####**Controlled starting conditions with torch.manual_seed()**
If I rerun this notebook tomorrow, how do I make sure my experiment starts from the same initial global model rather than a different random initialization?

Or simply, if we rerun the experiment, PyTorch can give the global model the same random starting weights, instead of a different initialization each time.

When we create:

**global_model = MyModel()**

PyTorch initializes the weights randomly. So rerunning the notebook can give the global model different initial weights.

In [3]:
torch.manual_seed(42)
global_model = MyModel()
client1 = MyModel()
client2 = MyModel()

When we write:

global_model = MyModel()

PyTorch gives the model random starting weights.

So if we run the notebook again, we may get different starting weights.

If we write:

torch.manual_seed(42)
global_model = MyModel()

We are basically telling PyTorch:

“Use the same random sequence as before.”

So every time we rerun that code, the model starts with the same random weights.

In [4]:
# extract the global model’s state into a variable
# load that state into client1 and client2.
global_model_state = global_model.state_dict()
client1.load_state_dict(global_model_state)
client2.load_state_dict(global_model_state)


<All keys matched successfully>

**Independent/Different model objects**


client1 and client2 must be different Python objects, so training one does not change the other.

In [5]:
# verify the object independence
print(client1 is global_model)
print(client2 is client1)
print(client2 is global_model)

False
False
False


**Same/Identical starting parameter values**


Corresponding parameters must initially contain the same numerical values

In [6]:
torch.equal(client1.state_dict()["layer1.weight"], client2.state_dict()["layer1.weight"])

True

#####**Saving Checkpoints**
Suppose we train for 50 epochs and stop.

If we save only:  **model.state_dict()**, we preserve the model’s learned parameter values.

That is enough if our only goal is:

 ***Load this trained model later and use it***

But if our goal is:

***Resume training from exactly where I stopped***

then the model parameters are only part of the story.

We also need to know things like:

1. model state      → what the model has learned
2. optimizer state  → how the optimizer has been updating the model
3. epoch            → where training stopped

In [7]:
# We build a simple optimizer for global_model
# which helps us understand how to build and save the checkpoint
optimiser = torch.optim.SGD(global_model.parameters(), lr = 0.01)
epoch = 50

**Why the optimizer state matters:**

Some optimizers, such as Adam, keep internal information from previous updates. If we reload only the model parameters but create a fresh optimizer, the model weights are restored, but the optimizer's training history is lost.

In [10]:
checkpoint = {
    "model_state": global_model.state_dict(),
    "optimiser_state": optimiser.state_dict(),
    "epoch": epoch
}

A checkpoint is usually stored as a Python dictionary.

The strings like "model_state", "optmiser_state", epoch are just labels, so we can retrieve each piece later.

In [11]:
# Save the object called checkpoint into a file called checkpoint.pth
torch.save(checkpoint, "checkpoint.pth")

#####**Loading and restoring a checkpoint**

In [15]:
#bring the saved dictionary back from the file into Python
loaded_checkpoint = torch.load("checkpoint.pth")
print(loaded_checkpoint.keys())

dict_keys(['model_state', 'optimiser_state', 'epoch'])


Now we restore them one by one.

First, create a fresh model and a fresh optimizer attached to that new model:

In [19]:
restored_model = MyModel()
restored_optimiser = torch.optim.SGD(restored_model.parameters(),
                                     lr = 0.01)
restored_model.load_state_dict(loaded_checkpoint["model_state"])

<All keys matched successfully>

In [22]:
restored_optimiser.load_state_dict(loaded_checkpoint["optimiser_state"])
restored_epoch = loaded_checkpoint["epoch"]

In [23]:
torch.equal(global_model.state_dict()["layer1.weight"], restored_model.state_dict()["layer1.weight"])

True

state_dict() gives me the model state organised by parameter/buffer names, so we can access and compare a specific saved tensor by its key.

#####**map location**

This matters when a checkpoint was saved on one device and loaded on another.

For example:

saved on GPU -> loaded later on CPU

In [ ]:
# tells PyTorch which device should hold the tensors
# when the checkpoint is loaded.
loaded_checkpoint = torch.load(
    "checkpoint.pth",
    map_location="cpu"
)

```text
Need multiple clients
        ↓
They must start identically
but train independently
        ↓
Different objects + same starting state
        ↓
deepcopy()
or
new model + load_state_dict()
        ↓
manual_seed()
for reproducible initialization
        ↓
checkpoint
= model state + optimizer state + training position
        ↓
torch.save()
        ↓
torch.load()
        ↓
restore each piece to the object it belongs to
```